In [0]:
# === Init cell with config (RUN FIRST) ===

storage_account_hierarchical = "foremstorageaccount"
bronze_adsl_path = f"abfss://adf-bronze@{storage_account_hierarchical}.dfs.core.windows.net/"
silver_adsl_path = f"abfss://adf-silver@{storage_account_hierarchical}.dfs.core.windows.net/"

df_bronze = spark.read.format("delta").load(bronze_adsl_path)
df_bronze.printSchema()

In [0]:
# === BATCH MERGE STRATEGY ===
# Read bronze, transform data batch and merge to silver

from delta.tables import DeltaTable
from pyspark.sql.functions import col, year, month, dayofmonth, to_date

pipeline_name = "silver"

# Processed data
processed_batches_folders = spark.sql(f"""
    SELECT batch_id
    FROM metadata.processed_batches
    WHERE pipeline_name = '{pipeline_name}'
    """)
processed_batches = [row.batch_id for row in processed_batches_folders.collect()]

current_batches = []

# Apply transformations to create df_silver
df_silver = df_bronze.filter(
    ~col("batch_id").isin(processed_batches)  # the `~` negates `isin()`
)

print(f"Bronze records batch2: {df_silver.count()}")

df_silver = df_silver.dropDuplicates(
    ["id"]
)
print(f"Bronze records batch2 unique rows: {df_silver.count()}")

df_silver = (
    df_silver.filter(
        (col("reading_time_minutes").isNotNull()) & (col("reading_time_minutes") != 0)
    )
    .select(
        "id",
        "title",
        "created_at",
        "published_at",
        "user",
        "tag_list",
        "positive_reactions_count",
        "public_reactions_count",
        "comments_count",
        "reading_time_minutes",
        "language",
        "collection_id",
        "batch_id",
        "batch_ingestion_date",
    )
    .withColumn("user_id", col("user.user_id"))
    .withColumn("user_username", col("user.username"))
    .withColumn("user_name", col("user.name"))
    .withColumn("date", to_date("published_at"))
    .withColumn("year", year("published_at"))
    .withColumn("month", month("published_at"))
    .withColumn("day", dayofmonth("published_at"))
)

print(f"Silver records after cleaning: {df_silver.count()}")

if DeltaTable.isDeltaTable(spark, silver_adsl_path):
    # Table exists - perform MERGE (upsert)
    silver_table = DeltaTable.forPath(spark, silver_adsl_path)

    silver_table.alias("target").merge(
        df_silver.alias("source"), "target.id = source.id"
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

    print(f"✓ Merged data into existing silver table")
else:
    # Table doesn't exist - create it with partitioning by year and month columns
    df_silver.write.format("delta").partitionBy("year", "month").mode("overwrite").save(
        silver_adsl_path
    )

    print(f"✓ Created new silver table with partitioning by year and month")

print(f"Silver table location: {silver_adsl_path}")

# Write metadata
print(f"Writing metadata for current batches: {current_batches}")
for current_batch in current_batches:
    batch_id = current_batch.rstrip("/").split("/")[-1]
    print('batch_id', batch_id)
    spark.sql(f"""
        MERGE INTO metadata.processed_batches t
        USING (
        SELECT
            'bronze' AS pipeline_name,
            '{batch_id}' AS batch_id,
            '{current_batch}' AS batch_folder,
            current_timestamp() AS processed_at
        ) s
        ON t.pipeline_name = s.pipeline_name
        AND t.batch_id = s.batch_id

        WHEN NOT MATCHED THEN
        INSERT (pipeline_name, batch_id, batch_folder, processed_at)
        VALUES (s.pipeline_name, s.batch_id, s.batch_folder, s.processed_at)
        """)